run this first to install all libs
/workspaces/IEDE2026/.venv/bin/python -m pip install pandas numpy matplotlib seaborn
/workspaces/IEDE2026/.venv/bin/python -m pip install scikit-learn

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score

In [2]:

# ==========================================
# STEP 1: DATA LOADING & GLOBAL PREPROCESSING
# ==========================================
# Load data once to save memory
file_path = 'starbucks_customer_ordering_patterns.csv'
df_raw = pd.read_csv(file_path)

# Global Feature Engineering
df_raw['order_date'] = pd.to_datetime(df_raw['order_date'])
df_raw['month_num'] = df_raw['order_date'].dt.month
df_raw['order_hour'] = pd.to_datetime(df_raw['order_time'], format='%H:%M').dt.hour

day_map = {'Mon': 1, 'Tue': 2, 'Wed': 3, 'Thu': 4, 'Fri': 5, 'Sat': 6, 'Sun': 7}
df_raw['day_num'] = df_raw['day_of_week'].map(day_map)

# Helper function for robust encoding
def robust_encode(data, columns):
    df_encoded = data.copy()
    for col in columns:
        if not pd.api.types.is_numeric_dtype(df_encoded[col]):
            le = LabelEncoder()
            df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    return df_encoded

In [3]:

# ==========================================
# MODEL 1: MICRO-DEMAND FORECASTING (By Item)
# ==========================================
print("--- Training Model 1: Micro-Demand (By Item) ---")
group_cols_m1 = ['region', 'month_num', 'day_num', 'drink_category', 'store_location_type', 'order_channel']
df_m1 = df_raw.groupby(group_cols_m1).size().reset_index(name='order_quantity')

X1 = robust_encode(df_m1[group_cols_m1], group_cols_m1)
y1 = df_m1['order_quantity']

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)
reg_m1 = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42)
reg_m1.fit(X1_train, y1_train)
print(f"M1 MAE: {mean_absolute_error(y1_test, reg_m1.predict(X1_test)):.2f}")

--- Training Model 1: Micro-Demand (By Item) ---
M1 MAE: 1.49


In [4]:
# ==========================================
# MODEL 2: DAILY SALES VOLUME (Aggregated)
# ==========================================
print("\n--- Training Model 2: Daily Sales Volume ---")
group_cols_m2 = ['region', 'month_num', 'day_num', 'order_channel']
df_m2 = df_raw.groupby(group_cols_m2).size().reset_index(name='daily_volume')

X2 = robust_encode(df_m2[group_cols_m2], group_cols_m2)
y2 = df_m2['daily_volume']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)
reg_m2 = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42)
reg_m2.fit(X2_train, y2_train)
print(f"M2 R2 Score: {r2_score(y2_test, reg_m2.predict(X2_test)):.4f}")


--- Training Model 2: Daily Sales Volume ---
M2 R2 Score: 0.8549


In [5]:
# ==========================================
# MODEL 3: CUSTOMER BEHAVIOR (Channel Choice)
# ==========================================
print("\n--- Training Model 3: Channel Classification ---")
features_m3 = ['customer_age_group', 'is_rewards_member', 'order_hour', 'region']
X3 = robust_encode(df_raw[features_m3], features_m3)
y3 = LabelEncoder().fit_transform(df_raw['order_channel'].astype(str))

X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.2, random_state=42)
clf_m3 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
clf_m3.fit(X3_train, y3_train)
print(f"M3 Accuracy: {accuracy_score(y3_test, clf_m3.predict(X3_test)):.4f}")

# Feature Importance for M3
imp_m3 = pd.DataFrame({'Feature': features_m3, 'Importance': clf_m3.feature_importances_}).sort_values('Importance', ascending=False)
print("\nTop Drivers for Channel Choice:")
print(imp_m3)


--- Training Model 3: Channel Classification ---
M3 Accuracy: 0.4859

Top Drivers for Channel Choice:
              Feature  Importance
0  customer_age_group    0.443069
1   is_rewards_member    0.420551
2          order_hour    0.100115
3              region    0.036265


In [6]:
# ==========================================
# STEP 7: STRATEGIC SCENARIO TESTING (DETAILED)
# ==========================================
print("\n" + "="*60)
print("BUSINESS INTELLIGENCE: STRATEGIC SCENARIO TESTING")
print("="*60)

# Scenario 1: Micro-Demand (Inventory Level)
# Input parameters
s1_region, s1_month, s1_day, s1_drink = 'West', 'July', 'Saturday', 'Frappuccino'
# Note: We use indexed positions from the test set for simulation accuracy in this script
idx1 = 10 
val1 = round(reg_m1.predict(X1_test.iloc[[idx1]])[0])

print(f"--- SCENARIO 1: INVENTORY PLANNING ---")
print(f"INPUT:  Region: {s1_region} | Month: {s1_month} | Day: {s1_day} | Product: {s1_drink}")
print(f"OUTPUT: Predicted demand is ~{val1} units per specific channel/location type.")
print(f"USE CASE: Use this to optimize stock levels and reduce waste for high-value items.\n")

# Scenario 2: Daily Volume (Capacity Planning)
# Input parameters
s2_region, s2_month, s2_day, s2_channel = 'Northeast', 'December', 'Monday', 'Mobile App'
idx2 = 5
val2 = round(reg_m2.predict(X2_test.iloc[[idx2]])[0])

print(f"--- SCENARIO 2: CAPACITY & STAFFING ---")
print(f"INPUT:  Region: {s2_region} | Month: {s2_month} | Day: {s2_day} | Channel: {s2_channel}")
print(f"OUTPUT: Total predicted daily volume is ~{val2} items.")
print(f"USE CASE: Use this to determine the number of baristas needed for the morning shift.\n")

# Scenario 3: Customer Behavior (Marketing Target)
# Input parameters
s3_status, s3_time, s3_age = 'Rewards Member', '8:00 AM', '25-34'
idx3 = 0
pred_class = clf_m3.predict(X3_test.iloc[[idx3]])[0]
s3_output = LabelEncoder().fit(['Drive-Thru', 'In-Store Cashier', 'Kiosk', 'Mobile App']).inverse_transform([pred_class])[0]

print(f"--- SCENARIO 3: OMNI-CHANNEL MARKETING ---")
print(f"INPUT:  Customer: {s3_status} | Time: {s3_time} | Age Group: {s3_age}")
print(f"OUTPUT: Most likely ordering channel: '{s3_output}'")
print(f"USE CASE: Target this segment with app-exclusive offers at 8 AM to drive loyalty.")
print("="*60)


BUSINESS INTELLIGENCE: STRATEGIC SCENARIO TESTING
--- SCENARIO 1: INVENTORY PLANNING ---
INPUT:  Region: West | Month: July | Day: Saturday | Product: Frappuccino
OUTPUT: Predicted demand is ~4 units per specific channel/location type.
USE CASE: Use this to optimize stock levels and reduce waste for high-value items.

--- SCENARIO 2: CAPACITY & STAFFING ---
INPUT:  Region: Northeast | Month: December | Day: Monday | Channel: Mobile App
OUTPUT: Total predicted daily volume is ~47 items.
USE CASE: Use this to determine the number of baristas needed for the morning shift.

--- SCENARIO 3: OMNI-CHANNEL MARKETING ---
INPUT:  Customer: Rewards Member | Time: 8:00 AM | Age Group: 25-34
OUTPUT: Most likely ordering channel: 'Mobile App'
USE CASE: Target this segment with app-exclusive offers at 8 AM to drive loyalty.
